In [1]:
import ifcopenshell
import ifcopenshell.api
import ifcopenshell.util
import ifcopenshell.api.root
import ifcopenshell.api.context
import ifcopenshell.api.aggregate

from saron.ifc_session import IfcSession

In [2]:
session = IfcSession()
session.open_ifc_project("/Users/anthonydemattos/syyclops/api/hyatt-mech.ifc")

In [4]:
tree = session.get_geometry_tree()
tree

'#1815 = IfcBuildingElementProxy "Cabinet_Unit_Heater-QMark-CU900:Not a Type - Load Type Catalog:1898042" (2yNlTM1i1A5hmjL3utI4aA)\n.  Cabinet_Unit_Heater-QMark-CU900:Not a Type - Load Type Catalog\n#12281 = IfcBuildingElementProxy "BVN_ARMO-A:ARMO-A 1250/6-20:1899544" (2yNlTM1i1A5hmjL3utI4Ce)\n.  BVN_ARMO-A:ARMO-A 1250/6-20\n#12342 = IfcBuildingElementProxy "BVN_ARMO-A:ARMO-A 1250/6-20:1899667" (2yNlTM1i1A5hmjL3utI4EZ)\n.  BVN_ARMO-A:ARMO-A 1250/6-20\n#12364 = IfcBuildingElementProxy "BVN_ARMO-A:ARMO-A 1250/6-20:1899774" (2yNlTM1i1A5hmjL3utI4FE)\n.  BVN_ARMO-A:ARMO-A 1250/6-20\n#12388 = IfcBuildingElementProxy "BVN_ARMO-A:ARMO-A 1250/6-20:1899868" (2yNlTM1i1A5hmjL3utI49i)\n.  BVN_ARMO-A:ARMO-A 1250/6-20\n#12412 = IfcBuildingElementProxy "BVN_ARMO-A:ARMO-A 1250/6-20:1899911" (2yNlTM1i1A5hmjL3utI4At)\n.  BVN_ARMO-A:ARMO-A 1250/6-20\n#12436 = IfcBuildingElementProxy "BVN_ARMO-A:ARMO-A 1250/6-20:1899919" (2yNlTM1i1A5hmjL3utI4A$)\n.  BVN_ARMO-A:ARMO-A 1250/6-20\n#12460 = IfcBuildingElement

In [4]:
project_library = session.ifc_project_library
bim_objects = project_library.Declares[0].RelatedDefinitions
tree = session.get_ifc_project_library_tree()
tree

'   * IfcBuildingElementProxyType\n      * Mobile Crane 50T\n      * Site Shed 3x12m\n      * Site Shed 3x6m\n'

In [5]:
model = ifcopenshell.file(schema="IFC4")

In [6]:
project = ifcopenshell.api.root.create_entity(model, ifc_class="IfcProject", name="My Project")

# Set up units - using millimeters
units = model.create_entity("IfcUnitAssignment")
length_unit = model.create_entity("IfcSIUnit", UnitType="LENGTHUNIT", Name="METRE")  # Remove the MILLI prefix
units.Units = [length_unit]
project.UnitsInContext = units

# Set up geometric representation contexts
context = model.create_entity(
    "IfcGeometricRepresentationContext",
    ContextType="Model",
    CoordinateSpaceDimension=3,
    Precision=0.01,
    WorldCoordinateSystem=model.create_entity(
        "IfcAxis2Placement3D",
        Location=model.create_entity("IfcCartesianPoint", Coordinates=(0.0, 0.0, 0.0)),
    ),
)

model_context = model.create_entity(
    "IfcGeometricRepresentationSubContext",
    ContextIdentifier="Body",
    ContextType="Model",
    ParentContext=context,
    TargetView="MODEL_VIEW",
)

# Create a site, building, and storey. Many hierarchies are possible.
site = ifcopenshell.api.root.create_entity(model, ifc_class="IfcSite", name="My Site")
building = ifcopenshell.api.root.create_entity(model, ifc_class="IfcBuilding", name="Building A")
storey = ifcopenshell.api.root.create_entity(model, ifc_class="IfcBuildingStorey", name="Ground Floor")

ifcopenshell.api.aggregate.assign_object(model, relating_object=project, products=[site])
ifcopenshell.api.aggregate.assign_object(model, relating_object=site, products=[building])
ifcopenshell.api.aggregate.assign_object(model, relating_object=building, products=[storey])

#13=IfcRelAggregates('28PewwRDzDTeD_juxRG9Po',$,$,$,#9,(#10))

In [7]:
# Get the crane type from library
crane_type = bim_objects[0]  # Assuming first object is the crane

In [8]:
# First copy all representation items and their styles
if crane_type.RepresentationMaps:
    for rep_map in crane_type.RepresentationMaps:
        # Copy all items in the representation
        for item in rep_map.MappedRepresentation.Items:
            # Copy styles if they exist
            if hasattr(item, 'StyledByItem'):
                for styled_item in item.StyledByItem:
                    # Copy the style assignment
                    model.add(styled_item)
                    for style in styled_item.Styles:
                        # Copy the presentation style
                        model.add(style)
                        if hasattr(style, 'Styles'):
                            for substyle in style.Styles:
                                # Copy surface styles and colors
                                model.add(substyle)
                                if hasattr(substyle, 'SurfaceColour'):
                                    model.add(substyle.SurfaceColour)

In [9]:
# Copy the crane type
new_crane_type = model.add(crane_type)

In [10]:
# Copy the representation maps and their contents
if crane_type.RepresentationMaps:
    for rep_map in crane_type.RepresentationMaps:
        model.add(rep_map)
        model.add(rep_map.MappedRepresentation)

# Create crane instance
crane = model.create_entity("IfcBuildingElementProxy", Name="cool crane")
crane.ObjectType = new_crane_type.Name

In [11]:
# Create type relationship
type_relationship = model.create_entity(
    "IfcRelDefinesByType",
    GlobalId=ifcopenshell.guid.new(),
    RelatedObjects=[crane],
    RelatingType=new_crane_type
)

# Create placement for the crane
placement = model.create_entity(
    "IfcLocalPlacement",
    PlacementRelTo=storey.ObjectPlacement,
    RelativePlacement=model.create_entity(
        "IfcAxis2Placement3D",
        Location=model.create_entity(
            "IfcCartesianPoint",
            Coordinates=(0.0, 0.0, 0.0)
        )
    )
)
crane.ObjectPlacement = placement

In [12]:
# Copy representation
if new_crane_type.RepresentationMaps:
    shape = model.create_entity(
        "IfcShapeRepresentation",
        ContextOfItems=model_context,
        RepresentationIdentifier=new_crane_type.RepresentationMaps[0].MappedRepresentation.RepresentationIdentifier,
        RepresentationType=new_crane_type.RepresentationMaps[0].MappedRepresentation.RepresentationType,
    )
    
    # Create mapping
    mapped_item = model.create_entity(
        "IfcMappedItem",
        MappingSource=new_crane_type.RepresentationMaps[0],
        MappingTarget=model.create_entity(
            "IfcCartesianTransformationOperator3D",
            Axis1=None,
            Axis2=None,
            LocalOrigin=model.create_entity(
                "IfcCartesianPoint",
                Coordinates=(0.0, 0.0, 0.0)
            ),
            Scale=1.0,
            Axis3=None
        )
    )
    shape.Items = [mapped_item]
    
    # Create product definition shape
    product_shape = model.create_entity(
        "IfcProductDefinitionShape",
        Representations=[shape]
    )
    crane.Representation = product_shape

In [13]:
# Assign the crane to the storey
ifcopenshell.api.aggregate.assign_object(
    model,
    relating_object=storey,
    products=[crane]
)

#231=IfcRelAggregates('0ef3HkueP0TB4mxOBQG18q',$,$,$,#10,(#221))

In [14]:
# Copy materials and their associations
if hasattr(crane_type, 'HasAssociations'):
    for association in crane_type.HasAssociations:
        if association.is_a('IfcRelAssociatesMaterial'):
            # Copy the material
            material = association.RelatingMaterial
            model.add(material)
            
            # Copy material properties
            if material.is_a('IfcMaterial'):
                if hasattr(material, 'HasProperties'):
                    for props in material.HasProperties:
                        model.add(props)
                if hasattr(material, 'HasRepresentation'):
                    model.add(material.HasRepresentation)
                    for rep in material.HasRepresentation.Representations:
                        model.add(rep)
            
            # Copy the association
            model.add(association)

In [15]:
# Write the new file
model.write("test.ifc")